<a href="https://colab.research.google.com/github/adithya-ashoks/deep-learning-practice/blob/main/keras_hyper_parameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

In [4]:
df = pd. read_csv("pima-indians-diabetes.csv")

In [5]:
df.head()

,pregnant,glucose,bp,skin,insulin,bmi,pedigree,age,label
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
df.corr()['label']

,label
pregnant,0.221898
glucose,0.466581
bp,0.065068
skin,0.074752
insulin,0.130548
bmi,0.292695
pedigree,0.173844
age,0.238356
label,1.000000


In [7]:
x = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x = scaler.fit_transform(x)

In [9]:
x.shape

(768, 8)

In [10]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=1)

In [71]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Dropout


In [12]:
model = Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [14]:
model.fit(x_train,y_train, batch_size=32,epochs=10, validation_data=(x_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.5179 - loss: 0.7452 - val_accuracy: 0.5909 - val_loss: 0.7086
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5961 - loss: 0.6915 - val_accuracy: 0.6299 - val_loss: 0.6587
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6238 - loss: 0.6497 - val_accuracy: 0.6753 - val_loss: 0.6206
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6466 - loss: 0.6177 - val_accuracy: 0.7013 - val_loss: 0.5892
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6645 - loss: 0.5898 - val_accuracy: 0.7143 - val_loss: 0.5635
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6840 - loss: 0.5659 - val_accuracy: 0.7143 - val_loss: 0.5420
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6954 - loss: 0.5471 - val_accuracy: 0.7403 - val_loss: 0.5243
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7215 - loss: 0.5307 - val_accuracy: 0.7662 - val_loss

In [15]:
#1. selecting appropriating optimizer
#2. no of nodes in a layer
#3. how to slect no of layers
#4. all in one model

In [17]:
import keras_tuner as kt

In [18]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(32,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  optimizer = hp.Choice('optimizer',values = ['adam', 'sgd', 'rmsprop', 'adadelta' ])
  model.compile(optimizer = optimizer ,loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [19]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy',max_trials=5)

In [20]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.6298701167106628

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 18s


In [21]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [22]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.7085 - loss: 0.6091 - val_accuracy: 0.7922 - val_loss: 0.5629
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7264 - loss: 0.5703 - val_accuracy: 0.7857 - val_loss: 0.5318
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7410 - loss: 0.5433 - val_accuracy: 0.7727 - val_loss: 0.5100
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7524 - loss: 0.5226 - val_accuracy: 0.7792 - val_loss: 0.4973
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7638 - loss: 0.5086 - val_accuracy: 0.7922 - val_loss: 0.4873
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7655 - loss: 0.4970 - val_accuracy: 0.8052 - val_loss: 0.4797
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7671 - loss: 0.4889 - val_accuracy: 0.8117 - val_loss: 0.4737
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7704 - loss: 0.4817 - val_accurac

In [46]:
def build_model(hp):
  model = Sequential()

  units = hp.Int('units', min_value=8, max_value=128, step=1)

  model.add(Dense(units=units,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [47]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy',max_trials=5, directory = 'mydir',project_name = 'proj2')

Reloading Tuner from mydir/proj2/tuner0.json


In [48]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

In [49]:
tuner.results_summary()

Results summary
Results in mydir/proj2
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
units: 48
Score: 0.7792207598686218

Trial 3 summary
Hyperparameters:
units: 64
Score: 0.7662337422370911

Trial 0 summary
Hyperparameters:
units: 16
Score: 0.7532467246055603

Trial 2 summary
Hyperparameters:
units: 24
Score: 0.7467532753944397

Trial 4 summary
Hyperparameters:
units: 8
Score: 0.5


In [50]:
tuner.get_best_hyperparameters()[0].values

{'units': 48}

In [51]:
model = tuner.get_best_models(num_models=1)[0]

In [52]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7573 - loss: 0.5311 - val_accuracy: 0.7662 - val_loss: 0.5090
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7655 - loss: 0.5101 - val_accuracy: 0.7727 - val_loss: 0.4942
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7704 - loss: 0.4972 - val_accuracy: 0.7662 - val_loss: 0.4844
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7785 - loss: 0.4870 - val_accuracy: 0.7727 - val_loss: 0.4770
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7818 - loss: 0.4790 - val_accuracy: 0.7662 - val_loss: 0.4748
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7834 - loss: 0.4737 - val_accuracy: 0.7597 - val_loss: 0.4717
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4689 - val_accuracy: 0.7662 - val_loss: 0.4714
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7818 - loss: 0.4652 - val_accuracy: 0.7

In [55]:
def build_model(hp) :
  model = Sequential()

  model.add(Dense(72,activation='relu',input_dim=8))

  for i in range(hp.Int('num_layers',min_value=1,max_value=10)) :
    model.add(Dense(72,activation='relu'))

  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [56]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy',max_trials=3, directory = 'mydir',project_name = 'num_layers')

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [57]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

Trial 3 Complete [00h 00m 06s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8441558480262756
Total elapsed time: 00h 00m 17s


In [58]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 34 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [60]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.7818 - loss: 0.4656 - val_accuracy: 0.7987 - val_loss: 0.5096
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7899 - loss: 0.4339 - val_accuracy: 0.7987 - val_loss: 0.4934
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7899 - loss: 0.4276 - val_accuracy: 0.7987 - val_loss: 0.4794
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8013 - loss: 0.4248 - val_accuracy: 0.7792 - val_loss: 0.4863
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8143 - loss: 0.4061 - val_accuracy: 0.7857 - val_loss: 0.4862
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8143 - loss: 0.3964 - val_accuracy: 0.7727 - val_loss: 0.4953
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8290 - loss: 0.3861 - val_accuracy: 0.7662 - val_loss: 0.4916
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8420 - loss: 0.3625 - val_accuracy: 0.75

In [78]:
def build_model(hp):
    model = Sequential()

    # Dynamically tune number of layers and units/activation per layer
    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
        model.add(
            Dense(
                units=hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation_' + str(i), values=['relu', 'tanh', 'sigmoid'])
            )
        )

        # Optional: Add dropout after hidden layers
        if hp.Boolean('dropout_' + str(i)):
            model.add(Dropout(rate=hp.Float('dropout_rate_' + str(i), min_value=0.1, max_value=0.5, step=0.1)))

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile model with dynamic optimizer
    model.compile(
        optimizer=hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta', 'nadam']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [79]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5, directory='mydir', project_name='final')

Reloading Tuner from mydir/final/tuner0.json


In [80]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [81]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 4,
 'units_0': 72,
 'activation_0': 'relu',
 'optimizer': 'rmsprop',
 'units_1': 16,
 'activation_1': 'relu',
 'units_2': 48,
 'activation_2': 'sigmoid',
 'units_3': 8,
 'activation_3': 'relu'}

In [82]:
model = tuner.get_best_models(num_models=1)[0]

In [83]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7687 - loss: 0.5149 - val_accuracy: 0.7857 - val_loss: 0.4892
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7752 - loss: 0.4909 - val_accuracy: 0.7922 - val_loss: 0.4738
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4788 - val_accuracy: 0.7987 - val_loss: 0.4662
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7671 - loss: 0.4715 - val_accuracy: 0.7987 - val_loss: 0.4618
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4645 - val_accuracy: 0.8052 - val_loss: 0.4582
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4630 - val_accuracy: 0.7857 - val_loss: 0.4593
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4591 - val_accuracy: 0.7857 - val_loss: 0.4591
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4563 - val_accuracy: 0.78